# Scenario-0 reactance, the phantom cycle, and how big it gets

**Companion to `ac_lines_kvl_check.ipynb`.** That notebook established *how* the
stochastic KVL behaves (Findings F1-F6). This one answers the quantitative
questions:

1. In the **real pypsa-de workflow**, does scenario ordering matter at all?
2. For a corridor with **zero / moderate / high** capacity across three
   scenarios, how far does the stochastic dispatch drift from a deterministic
   dispatch with the *physically correct* reactances?
3. How large is the effect on the **real 27-cluster grid**, and does the
   "strongest grid first" idea help there?
4. How might it scale to 1 h resolution, and what is the best mitigation?

**Headline.** In the real workflow the per-scenario reactance is *shared* (the
CSV overrides touch only `s_nom`, never `x`), so the stochastic solve is exactly
**order-invariant** - "strongest grid first" is a no-op. The real error is the
**phantom cycle**: a line zeroed in one scenario keeps its built-line reactance,
so `s_nom=0` with finite `x` forces its endpoints to equal voltage angle - a
spurious constraint that distorts flows on every other line in its cycle. It is
always pessimistic, is largest for meshed and double-circuit corridors, and
cannot be fixed by ordering.

*Run with the project `pixi`/`uv` env. The real-grid sections read one solved
27-cluster network that ships in `results/`; a resolver finds it automatically.*


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pypsa

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
print("pypsa", pypsa.__version__)

# colourblind-safe palette (Okabe-Ito), reused across figures
CB = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73",
      "red": "#D55E00", "purple": "#CC79A7", "grey": "#999999", "black": "#000000"}

## 0. The reframing: reactance is shared across scenarios

`ac_lines_kvl_check.ipynb` deliberately set a *different* `x` per scenario to
expose that the KVL weights follow `scenarios[0]` (F3). The real workflow never
does that. `build_grid_scenario_csvs.py` writes **only the capacity column**
(`s_nom`/`p_nom`); `build_grid_topology.py::_apply_overrides` applies it and sets
`s_nom_extendable=False`. Reactance `x` is left at the base (fully-built) value
in every scenario slice.

Below, `x` is the **series reactance in Ohm** (`n.lines["x"]`), not a coordinate
- coordinates live on buses (`n.buses["x"]` is longitude). We confirm on a real
solved stochastic network that the toggling lines carry an identical `x` in both
scenarios, and that the weighted cycle matrix has no scenario axis.


In [ ]:
ROOT = next((q for q in [Path.cwd(), *Path.cwd().parents] if (q / "results").exists()), Path.cwd())

STOCH = ROOT / ("results/20260921_merge_main_test-49cl-365h-2025EXO-vs-2035EXO-50-50/"
                "KN2045_Mix/networks/base_s_49__none_2035_topology-stochastic.nc")
ns = pypsa.Network(STOCH)
snom = ns.lines["s_nom"].unstack("scenario")
x = ns.lines["x"].unstack("scenario")
toggling = snom.index[snom.std(axis=1) > 1e-6]
print("scenarios:", list(ns.scenarios))
print("lines whose s_nom toggles across scenarios:", len(toggling))
print("max x std across scenarios among those lines:", float(x.loc[toggling].std(axis=1).max()))
print()
print("example (s_nom then x per scenario):")
print(snom.loc[toggling].head(3).round(0))
print(x.loc[toggling].head(3).round(4))

In [ ]:
# the weighted cycle matrix (reactance weights) has NO scenario axis -> one shared set
Cm = ns.cycle_matrix(apply_weights=True)
print("cycle_matrix index names:", Cm.index.names, "| column level:", Cm.columns.names)
print("shape (branches x cycles):", Cm.shape, "-> a single matrix for the whole stochastic net")

**Consequence - order-invariance.** A stochastic program is a sum over
scenarios, so it is inherently order-independent; the *only* thing that makes
PyPSA order-dependent is that it reads the KVL from `scenarios[0]`. With `x`
(and `bus0/bus1`) identical across scenarios, `scenarios[0]`'s weights equal
every scenario's, so even that dependence vanishes. The toy below confirms a
shared-`x` stochastic solve is bit-identical under any ordering.


In [ ]:
def toy_order(order, spec=None):
    """3-bus loop; swing capacity varies by scenario, x shared (workflow reality)."""
    spec = spec or {"zero": 0.0, "mod": 60.0, "high": 120.0}
    m = pypsa.Network(); m.set_snapshots([0])
    for b in ("1", "2", "3"):
        m.add("Bus", b, v_nom=380.0)
    m.add("Generator", "cheap", bus="1", p_nom=1000, marginal_cost=1.0)
    m.add("Generator", "backup", bus="3", p_nom=1000, marginal_cost=100.0)
    m.add("Load", "load", bus="3", p_set=100.0)
    m.add("Line", "swing", bus0="1", bus1="3", x=1.0, r=0.0, s_nom=0.0)   # x SHARED = 1
    m.add("Line", "det_a", bus0="1", bus1="2", x=1.0, r=0.0, s_nom=50.0)
    m.add("Line", "det_b", bus0="2", bus1="3", x=1.0, r=0.0, s_nom=50.0)
    m.set_scenarios({sc: 1 / len(order) for sc in order})
    for sc in order:
        m.lines.loc[(sc, "swing"), "s_nom"] = spec[sc]
    m.optimize(log_to_console=False, solver_name="highs")
    return m.objective

for order in (["zero", "mod", "high"], ["high", "mod", "zero"], ["mod", "high", "zero"]):
    print(f"order={order}  stochastic objective={toy_order(order):.4f}")
print("-> identical: scenario ordering is a no-op when x is shared (the workflow case).")

## 1. Three scenarios (zero / moderate / high): three "worlds"

A 3-bus loop isolates the effect. Cheap generation at bus 1 (1 EUR/MWh) serves a
100 MW load at bus 3 over a **direct** corridor (`swing`, bus 1-3) or a capped
**detour** `1-2-3` (two 50 MW lines). The swing corridor is reinforced by `N`
parallel circuits: capacity `N x 30 MW` and - *physically* - reactance `x0 / N`.
The scenarios are:

| scenario | circuits N | capacity | correct reactance |
|---|---|---|---|
| `zero` | 0 | 0 MW | `inf` (absent) |
| `mod` | 2 | 60 MW | `x0 / 2` |
| `high` | 4 | 120 MW | `x0 / 4` |

The base/canvas network is the fully-built grid, so its shared reactance is
`x_high = x0 / 4`. We compare three worlds:

- **A - as-is stochastic:** every scenario uses the shared `x_high` (reality).
- **B - per-topology deterministic:** each topology solved alone with its
  *correct* reactance - the ground truth.
- **C - scaled-x stochastic:** we set each scenario its correct `x`, but PyPSA
  still applies `scenarios[0]`'s weights to all (F3) - so this depends on which
  scenario is first.


In [ ]:
CAP1, X1 = 30.0, 4.0
COST = {"cheap": 1.0, "backup": 100.0}
SPEC = {"zero": (0.0, np.inf), "mod": (2 * CAP1, X1 / 2), "high": (4 * CAP1, X1 / 4)}
X_CANVAS = SPEC["high"][1]  # fully-built reactance, shared by the as-is workflow


def base_net():
    m = pypsa.Network(); m.set_snapshots([0])
    for b in ("1", "2", "3"):
        m.add("Bus", b, v_nom=380.0)
    m.add("Generator", "cheap", bus="1", p_nom=1000, marginal_cost=COST["cheap"])
    m.add("Generator", "backup", bus="3", p_nom=1000, marginal_cost=COST["backup"])
    m.add("Load", "load", bus="3", p_set=100.0)
    m.add("Line", "det_a", bus0="1", bus1="2", x=1.0, r=0.0, s_nom=50.0)
    m.add("Line", "det_b", bus0="2", bus1="3", x=1.0, r=0.0, s_nom=50.0)
    return m


def deterministic(sc):
    """World B ground truth: topology sc solved alone with its correct (cap, x)."""
    cap, x = SPEC[sc]
    m = base_net()
    if not np.isinf(x):
        m.add("Line", "swing", bus0="1", bus1="3", x=x, r=0.0, s_nom=cap)
    m.optimize(log_to_console=False, solver_name="highs")
    return m.objective


def stochastic(order, x_mode):
    """x_mode='canvas' -> shared x_high (World A); 'scaled' -> correct x per
    scenario, but scenarios[0] still governs the KVL for all (World C)."""
    first = order[0]
    x0 = X_CANVAS if x_mode == "canvas" else SPEC[first][1]
    m = base_net()
    m.add("Line", "swing", bus0="1", bus1="3", x=(np.inf if np.isinf(x0) else x0),
          r=0.0, s_nom=0.0)
    m.set_scenarios({sc: 1 / len(order) for sc in order})
    for sc in order:
        m.lines.loc[(sc, "swing"), "s_nom"] = SPEC[sc][0]
    m.optimize(log_to_console=False, solver_name="highs")
    costs = {}
    for sc in order:
        disp = m.generators_t.p.xs(sc, axis=1, level="scenario").loc[0]
        costs[sc] = sum(disp[g] * COST[g] for g in COST)
    return costs


det = {sc: deterministic(sc) for sc in SPEC}
E_correct = float(np.mean(list(det.values())))
print("World B (correct) per-scenario cost:", {k: round(v, 1) for k, v in det.items()})
print("World B E[cost]:", round(E_correct, 1))

In [ ]:
rows = []
# World A: as-is, two orderings (identical by construction)
for order in (["zero", "mod", "high"], ["high", "mod", "zero"]):
    c = stochastic(order, "canvas")
    rows.append({"world": "A as-is", "first": order[0], **{k: round(v, 1) for k, v in c.items()},
                 "E[cost]": round(np.mean(list(c.values())), 1)})
# World C: scaled x, each scenario first
for first in ("high", "mod", "zero"):
    order = [first] + [s for s in ("zero", "mod", "high") if s != first]
    c = stochastic(order, "scaled")
    rows.append({"world": "C scaled-x", "first": first, **{k: round(v, 1) for k, v in c.items()},
                 "E[cost]": round(np.mean(list(c.values())), 1)})
rows.append({"world": "B correct", "first": "per-topology",
             **{k: round(v, 1) for k, v in det.items()}, "E[cost]": round(E_correct, 1)})
tbl = pd.DataFrame(rows)[["world", "first", "zero", "mod", "high", "E[cost]"]]
tbl

**Read-off.**

- **World A is order-invariant** (both rows identical) and badly wrong: the
  `zero` scenario is a *phantom catastrophe* and the `mod` scenario carries a
  reactance mismatch; only `high` (= the canvas) is correct.
- **"Strongest grid first" = as-is.** The canvas already *is* the strongest grid,
  so putting it first changes nothing. The phantom in the weak scenarios stays.
- **Only "weakest-first" (`scenarios[0]=zero`, `x=inf`) recovers the correct
  expected cost** - by dropping the swing from the shared KVL entirely, i.e.
  modelling it as a controllable link. It is right *here* but over-idealises AC
  physics (free routing).
- A *large-but-finite* `x` does **not** remove the phantom; only true `inf` does.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
labels = ["A as-is\n(any order)", "C scaled\nhigh-first", "C scaled\nmod-first",
          "C scaled\nweakest-first", "B correct\n(ground truth)"]
vals = [tbl.loc[0, "E[cost]"],
        tbl[(tbl.world == "C scaled-x") & (tbl.first == "high")]["E[cost]"].iloc[0],
        tbl[(tbl.world == "C scaled-x") & (tbl.first == "mod")]["E[cost]"].iloc[0],
        tbl[(tbl.world == "C scaled-x") & (tbl.first == "zero")]["E[cost]"].iloc[0],
        E_correct]
colors = [CB["red"], CB["red"], CB["orange"], CB["green"], CB["blue"]]
ax.bar(labels, vals, color=colors)
ax.axhline(E_correct, ls="--", lw=0.8, color=CB["grey"])
ax.set_ylabel("expected system cost (EUR)")
ax.set_title("Only weakest-first (link) recovers the correct cost; ordering among\nfinite-x worlds never does")
for i, v in enumerate(vals):
    ax.annotate(f"{v:.0f}", (i, v), ha="center", va="bottom", fontsize=9)
fig

## 2. When does the phantom bite? Three regimes

The phantom only exists when the toggled line sits in a **cycle**. Deterministic
comparison (isolates the phantom): a zeroed line kept in the KVL (`s_nom=0`,
finite `x`) vs the line truly removed (`x=inf`). Load shedding is priced at
1000 EUR/MWh.


In [ ]:
VOLL = 1000.0

def solve(build, mode):
    m = build(mode); m.optimize(log_to_console=False, solver_name="highs")
    return m.objective

def bridge(mode):               # radial: swing is the only link, in no cycle
    m = pypsa.Network(); m.set_snapshots([0]); m.add("Bus", ["1", "2"], v_nom=380.0)
    m.add("Generator", "cheap", bus="1", p_nom=1000, marginal_cost=1.0)
    m.add("Generator", "shed", bus="2", p_nom=1000, marginal_cost=VOLL)
    m.add("Load", "load", bus="2", p_set=100.0)
    if mode != "removed": m.add("Line", "swing", bus0="1", bus1="2", x=1.0, r=0.0, s_nom=0.0)
    return m

def mesh(mode):                 # one path of a loop, capped detour alternative
    m = pypsa.Network(); m.set_snapshots([0]); m.add("Bus", ["1", "2", "3"], v_nom=380.0)
    m.add("Generator", "cheap", bus="1", p_nom=1000, marginal_cost=1.0)
    m.add("Generator", "shed", bus="3", p_nom=1000, marginal_cost=VOLL)
    m.add("Load", "load", bus="3", p_set=100.0)
    m.add("Line", "det_a", bus0="1", bus1="2", x=1.0, r=0.0, s_nom=50.0)
    m.add("Line", "det_b", bus0="2", bus1="3", x=1.0, r=0.0, s_nom=50.0)
    if mode != "removed": m.add("Line", "swing", bus0="1", bus1="3", x=1.0, r=0.0, s_nom=0.0)
    return m

def parallel(mode):             # parallel to an existing circuit (reinforcement)
    m = pypsa.Network(); m.set_snapshots([0]); m.add("Bus", ["1", "3"], v_nom=380.0)
    m.add("Generator", "cheap", bus="1", p_nom=1000, marginal_cost=1.0)
    m.add("Generator", "shed", bus="3", p_nom=1000, marginal_cost=VOLL)
    m.add("Load", "load", bus="3", p_set=100.0)
    m.add("Line", "existing", bus0="1", bus1="3", x=1.0, r=0.0, s_nom=100.0)
    if mode != "removed": m.add("Line", "swing", bus0="1", bus1="3", x=1.0, r=0.0, s_nom=0.0)
    return m

reg = pd.DataFrame([
    {"regime": nm, "phantom (s_nom=0)": solve(b, "phantom"),
     "correct (x=inf)": solve(b, "removed"),
     "phantom error": solve(b, "phantom") - solve(b, "removed")}
    for nm, b in [("bridge", bridge), ("mesh", mesh), ("parallel-reinforce", parallel)]
]).set_index("regime")
reg

**Taxonomy.** A **bridge** (radial) line is in no cycle, so `s_nom=0` is
harmless (phantom error 0). A **mesh** line distorts the alternative path. A
**parallel-reinforce** line is the worst: zeroing one circuit angle-locks the
parallel circuit to zero too, blocking the whole corridor. Real transmission
corridors are frequently double circuits - the catastrophic case.


## 3. The real 27-cluster grid

The saved sector-coupled networks (PyPSA 1.1.2) do not re-solve under 1.3.0, so
instead of re-optimising the full system we isolate the **AC grid physics** on
the real topology. From one solved 27-cluster network we read the stored AC line
flows (no re-solve), reconstruct the feasible nodal injections (lossless flow
divergence), and rebuild a *fresh* AC-only network with the **real reactances,
capacities (`s_nom_opt`) and security margins (`s_max_pu`)**.

On that testbed we measure the **redispatch / backup energy** forced by toggling
a corridor: `correct` (removed, `x=inf`) vs `phantom` (`s_nom=0`, real `x`).
Curtailment is free; local backup is priced at 1 EUR/MWh so the objective reads
directly as backup energy (MWh). *This is a flow-disruption proxy and an upper
bound on cost - it forbids cheap generation redispatch - but the direction and
flow pattern are faithful.*


In [ ]:
SRC = ROOT / "results/202600717-stoch-grid-testing-2030-70h/KN2045_Mix/networks/base_s_27__none_2030.nc"


def load_ac(path=SRC):
    s = pypsa.Network(path)
    ac = s.buses.index[s.buses.carrier == "AC"]
    L = s.lines
    p0 = s.lines_t.p0
    inj = pd.DataFrame(0.0, index=s.snapshots, columns=ac)
    for l in L.index:                       # lossless divergence (p0 only)
        inj[L.at[l, "bus0"]] += p0[l]
        inj[L.at[l, "bus1"]] -= p0[l]
    n = pypsa.Network()
    n.set_snapshots(s.snapshots)
    n.snapshot_weightings.loc[:, :] = s.snapshot_weightings.values
    n.add("Bus", ac, v_nom=s.buses.loc[ac, "v_nom"].values,
          country=s.buses.loc[ac, "country"].values)
    n.add("Line", L.index, bus0=L.bus0.values, bus1=L.bus1.values,
          x=L.x.values, r=L.r.values, s_nom=L.s_nom_opt.values, s_max_pu=L.s_max_pu.values)
    n.add("Load", "ld_" + ac, bus=ac)
    n.loads_t.p_set = (-inj).rename(columns=lambda b: "ld_" + b)
    n.add("Generator", "up_" + ac, bus=ac, p_nom=1e6, marginal_cost=1.0, sign=1.0)   # backup
    n.add("Generator", "dn_" + ac, bus=ac, p_nom=1e6, marginal_cost=0.0, sign=-1.0)  # free curtail
    return n, L, s


def backup(n, toggle=None, mode=None):
    m = n.copy()
    if toggle is not None:
        if mode == "correct": m.remove("Line", toggle)
        elif mode == "phantom": m.lines.at[toggle, "s_nom"] = 0.0
    m.optimize(log_to_console=False, solver_name="highs")
    up = m.generators_t.p.filter(like="up_").mul(m.snapshot_weightings.objective, axis=0)
    return up.values.sum()


n, L, s = load_ac()
w = s.snapshot_weightings.objective
print("AC buses:", len(n.buses), "lines:", len(n.lines), "snapshots:", len(n.snapshots))
print("BASE backup (all lines):", round(backup(n), 1), "MWh  (expect 0)")
thru = pd.DataFrame({b: 0.0 for b in n.buses.index}, index=s.snapshots)
print("DE AC grid throughput ~", round((s.lines_t.p0.abs().mul(w, axis=0)).values.sum() / 1e6, 0),
      "TWh/yr total AC transport (context for the MWh below)")

In [ ]:
# scan every DE-internal AC corridor
cty = s.buses.country
de = L.index[(L.bus0.map(cty) == "DE") & (L.bus1.map(cty) == "DE")]
Cm27 = s.cycle_matrix()
incyc = {nm for (t, nm), r in Cm27.iterrows() if t == "Line" and (r != 0).any()}

scan = []
for l in de:
    ph = backup(n, l, "phantom"); co = backup(n, l, "correct")
    scan.append({"line": l, "s_nom": L.at[l, "s_nom_opt"], "x": L.at[l, "x"],
                 "in_cycle": l in incyc, "phantom_TWh": ph / 1e6,
                 "correct_TWh": co / 1e6, "error_TWh": (ph - co) / 1e6})
scan = pd.DataFrame(scan).set_index("line").sort_values("error_TWh", ascending=False)
print(scan.round(2).to_string())
print("\nall", len(scan), "DE cycle corridors show phantom > correct (always pessimistic)")

In [ ]:
# double-circuit corridors: pairs of lines with identical phantom cost
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
y = np.arange(len(scan))
ax.barh(y, scan.phantom_TWh, color=CB["red"], label="phantom (s_nom=0, real x)")
ax.barh(y, scan.correct_TWh, color=CB["blue"], label="correct (x=inf)")
ax.set_yticks(y); ax.set_yticklabels(scan.index)
ax.invert_yaxis()
ax.set_xlabel("forced backup / redispatch energy (TWh/yr)  -  disruption proxy")
ax.set_ylabel("DE AC corridor (line id)")
ax.set_title("Phantom vs correct, per DE corridor (27 clusters)")
ax.legend(loc="lower right")
fig

In [ ]:
# flow distortion from zeroing one representative corridor
LINE = "27"
def mean_flow(m):
    cols = [l for l in de if l in m.lines.index]
    return m.lines_t.p0[cols].abs().mul(w, axis=0).sum() / w.sum()

mp = n.copy(); mp.lines.at[LINE, "s_nom"] = 0.0; mp.optimize(log_to_console=False, solver_name="highs")
mc = n.copy(); mc.remove("Line", LINE); mc.optimize(log_to_console=False, solver_name="highs")
dist = (mean_flow(mp) - mean_flow(mc)).reindex(de).fillna(mean_flow(mp)).abs().sort_values(ascending=False)
print(f"zeroing line {LINE}: DE lines with >50 MW mean-flow distortion:", int((dist > 50).sum()), "of", len(de))
print(dist.head(8).round(0).to_string())

In [ ]:
# 3-scenario stochastic on the real grid: order-invariance + weakest-first
full = L.at[LINE, "s_nom_opt"]
SP = {"zero": 0.0, "mod": 0.5 * full, "high": full}

def real_stoch(order, weakest_first=False):
    m = n.copy()
    if weakest_first:
        order = ["zero"] + [o for o in order if o != "zero"]
    m.set_scenarios({sc: 1 / 3 for sc in order})
    for sc in order:
        m.lines.loc[(sc, LINE), "s_nom"] = SP[sc]
    if weakest_first:
        m.lines.loc[(order[0], LINE), "x"] = np.inf   # drop from shared KVL -> link
    m.optimize(log_to_console=False, solver_name="highs")
    up = m.generators_t.p.filter(like="up_")
    return {sc: up.xs(sc, axis=1, level="scenario").mul(w, axis=0).values.sum() / 1e6 for sc in order}

for order in (["zero", "mod", "high"], ["high", "mod", "zero"], ["mod", "zero", "high"]):
    per = real_stoch(order)
    print(f"as-is order={order}: E={np.mean(list(per.values())):.2f} TWh  per-sc={ {k: round(v,2) for k,v in per.items()} }")
wf = real_stoch(["zero", "mod", "high"], weakest_first=True)
print(f"weakest-first (x=inf): E={np.mean(list(wf.values())):.2f} TWh  per-sc={ {k: round(v,2) for k,v in wf.items()} }")
print("ground-truth zero (removed):", round(scan.loc[LINE, 'correct_TWh'], 2), "TWh")

**Real-grid findings.**

- **Order-invariance holds exactly** on the real grid: all three scenario
  orderings give identical expected backup. **"Strongest grid first" is a
  no-op**, as predicted.
- The phantom is pervasive: **every DE cycle corridor** shows phantom > correct,
  always pessimistic. Pairs of lines with identical cost are **double-circuit
  corridors** - the catastrophic parallel regime.
- Zeroing a single corridor distorts GW-scale flows on ~2/3 of the other DE
  corridors - the phantom is not local.
- **Weakest-first (`x=inf`)** cuts the expected error by ~85% and recovers the
  correct `zero`-scenario value exactly, by modelling the uncertain corridor as
  a link.

*Caveat on magnitude.* The backup/TWh numbers are a disruption proxy (they
forbid cheap generation redispatch) and so overstate literal cost; the real
co-optimised model would absorb much of it with cheaper redispatch. The **flow
distortion, the direction (always pessimistic), the order-invariance, and the
mitigation ranking are faithful**. The exact *cost* bound is given cleanly by the
toy (Section 1): up to ~100% of the corridor's value in adverse topologies.


## 4. How this scales to 1 h resolution (reasoning, not run)

The phantom is a **per-snapshot structural constraint** (`theta_i = theta_j`
whenever a cycle line is at `s_nom=0`). It applies in *every* hour regardless of
resolution, so its existence and direction are resolution-independent.

What changes going from 126 representative snapshots (~70 h each) to 8760 hourly
steps:

- **Worst-case bite grows.** Representative snapshots smooth away the most
  congested hours (winter peak load + low wind). Those are exactly the hours
  with the least redispatch headroom, where the phantom's spurious angle-lock is
  most binding. Hourly resolution resolves them, so the *relative* error is
  likely to **grow modestly**, not shrink. The 126-snapshot figure is a
  conservative (lower) estimate of the hourly disruption.
- **Normalised metrics stay stable.** Absolute MWh scale with the number of
  represented hours, but error as a *fraction* of demand/throughput should be
  comparable or slightly worse.
- **Investment feedback compounds.** With endogenous capacity the phantom's
  pessimism about the weak-grid scenario biases the shared here-and-now decision
  toward **over-building** generation/storage to serve spurious congestion. At
  1 h this feedback is stronger (more binding hours), and in a myopic pathway it
  accumulates year on year.

Net: 1 h resolution does not remove the problem and probably makes the
worst-case somewhat worse. The structural fix must come from the model
formulation, not the time resolution.


## 5. Mitigations, ranked

A crucial constraint shapes the ranking: **PyPSA's native stochastic KVL cannot
hold a per-scenario reactance.** Even if you write the physically correct `x`
into each scenario slice (`x0/N`, `inf` when removed), only `scenarios[0]`'s
weights are used (F3). So the "obvious" fix - scale `x` with `s_nom` - is **not
deliverable** inside one stochastic solve. The options that remain:

1. **Model uncertain corridors as controllable `Link`s** (or, equivalently, put
   the *absent* topology first with `x=inf` so the corridor is dropped from the
   shared KVL). *Removes the phantom entirely.* Cost: over-idealises AC physics
   (free routing) -> a small *optimistic* bias, far smaller than the phantom's
   large *pessimistic* one. **Best general-purpose fix** when the uncertain
   corridors are few and the weak-grid scenario matters. (Section 1 weakest-first;
   Section 3 confirms on the real grid.)
2. **Per-topology deterministic solves** with correct `x` per grid. *Exact
   physics*, but loses the joint here-and-now coupling. Use to **bound and
   validate** the stochastic run (it is the ground truth in Sections 1 and 3).
3. **Make only radial/bridge corridors uncertain.** If you can choose, toggle
   corridors that sit in no cycle - the phantom error there is exactly zero
   (Section 2). Check each toggling line against `n.cycle_matrix()`.
4. **Deliberate `scenarios[0]`** - only meaningful once combined with option 1
   (choosing the `x=inf`/link topology first). On its own, with shared `x`, it
   does nothing.
5. **Accept and document** the bias where corridors are few and meshed-but-not-
   parallel: it is bounded and always pessimistic (never optimistic), so results
   are conservative on the weak-grid scenario.

**Recommended.** Combine (1) + (2): model the handful of genuinely uncertain
corridors as links inside the stochastic solve, and run per-topology
deterministic solves alongside to bound the idealisation error. Avoid relying on
scenario ordering or on scaling `x` per scenario - neither works in native PyPSA.


## Verdict

1. In the real pypsa-de workflow the per-scenario reactance is **shared**
   (overrides touch only `s_nom`), so the stochastic solve is **exactly
   order-invariant** - *strongest grid first is a no-op* (Sections 0, 1, 3).
2. The real error is the **phantom cycle**: `s_nom=0` with a finite built-line
   `x` forces equal endpoint voltage angles, a spurious constraint that distorts
   flows on every other line in the cycle. Always pessimistic (Sections 1-3).
3. It is **worst for parallel/double-circuit and meshed corridors, zero for
   radial ones** (Section 2); on the 27-cluster grid every cycle corridor is
   affected and single-corridor zeroing distorts GW-scale flows elsewhere
   (Section 3).
4. **1 h resolution does not fix it** and likely worsens the worst case
   (Section 4).
5. The **best mitigation** is to model uncertain corridors as links (removing the
   phantom at the price of a small optimistic idealisation) and bound it with
   per-topology deterministic solves. Scaling `x` per scenario cannot help -
   PyPSA uses only `scenarios[0]`'s KVL weights (Section 5).
